# Проверка agr_id по логике `spb_rf_clients_apr_jun_tariff_selection`

Те же Excel (апр–май–июнь 2026) и те же правила:
- присутствие во всех 3 месяцах;
- стабильный `tariff_group`;
- для **Стандарт** / **Меню возможностей** — июньские фильтры;
- филиал содержит `Санкт-Петербургский`.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

In [ ]:
DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')

TARGET_RF = 'Санкт-Петербургский'
REG_DATE_MAX = pd.Timestamp('2026-01-01')
STANDARD_TRX_SUM_MAX = 400_000.0
MENU_TRX_SUM_MAX = 0.0
COMM_MONTHLY_EPS = 0.005
FILTER_MONTH = '2026-06'
MONTHS = ['2026-04', '2026-05', '2026-06']
TARIFF_ORDER = ['0', 'Меню возможностей', 'По Акту индивидуальный', 'Стандарт']

CHECK_AGR_IDS = [
    '496703167537',
    '413636181589',
    '508492576892',
    '818953761556',
    '296467554524',
    '296467554554',
    '509630192857',
    '436063002451',
]

excel_sources = [
    {'report_month': '2026-04-01', 'path': DATA_DIR / '04_Апрель_2026.xlsx', 'header': 0},
    {'report_month': '2026-05-01', 'path': DATA_DIR / '05_Май_2026.xlsx', 'header': 0},
    {'report_month': '2026-06-01', 'path': DATA_DIR / '06_Июнь_2026.xlsx', 'header': 0},
]

for src in excel_sources:
    p = Path(src['path'])
    print(f"{src['report_month'][:7]}: exists={p.exists()} | {p}")

In [ ]:
def normalize_colname(value):
    s = str(value).lower().replace('\n', ' ').replace('\r', ' ').replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.replace('₽', 'руб').replace('%', 'pct')
    s = re.sub(r'[^a-zа-я0-9]+', '', s)
    return s


def pick_column(columns, aliases):
    cols = list(columns)
    norm_map = {normalize_colname(c): c for c in cols}
    for alias in aliases:
        if alias in cols:
            return alias
        key = normalize_colname(alias)
        if key in norm_map:
            return norm_map[key]
    for alias in aliases:
        key = normalize_colname(alias)
        for nk, original in norm_map.items():
            if key and key in nk:
                return original
    return None


def to_num(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace('\xa0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False),
        errors='coerce',
    )


def normalize_agr_id(value):
    if pd.isna(value):
        return None
    s = str(value).strip().replace('\xa0', '').replace(' ', '')
    if s.lower() in {'', 'nan', 'none'}:
        return None
    s = re.sub(r'\.0$', '', s)
    return s


def normalize_inn(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else s


def classify_tariff(value):
    if pd.isna(value):
        return None
    raw = str(value).strip()
    if raw == '' or raw.lower() in {'nan', 'none'}:
        return None
    if re.fullmatch(r'0(\.0+)?', raw):
        return '0'
    t = raw.lower().replace('ё', 'е')
    t = re.sub(r'\s+', ' ', t).strip()
    has_akt = 'акт' in t
    has_ind = 'индивид' in t
    if has_akt and has_ind:
        return 'По Акту индивидуальный'
    if has_akt and ('по акту' in t or t.startswith('по акт')):
        return 'По Акту индивидуальный'
    if 'меню' in t or 'возможност' in t:
        return 'Меню возможностей'
    if 'стандарт' in t:
        return 'Стандарт'
    return None


def _num0(v):
    if pd.isna(v):
        return 0.0
    try:
        return float(v)
    except (TypeError, ValueError):
        return 0.0


def _is_closed(v):
    if pd.isna(v):
        return False
    dt = pd.to_datetime(v, errors='coerce')
    if pd.notna(dt):
        if dt.year <= 1901:
            return False
        return True
    s = str(v).strip().lower()
    return s not in {'', 'nan', 'none', 'nat', 'na', '<null>', 'null', '-', '0', '0.0'}


COL_ALIASES = {
    'agr_id': ['ID договора', 'agr_id', 'abs_agr_id'],
    'company_name': ['Наименование', 'Наименование клиента', 'Наименование эквайринга', 'company_name'],
    'inn': ['ИНН', 'ИНН клиента', 'inn', 'c_inn'],
    'contract_number': ['Номер договора', 'Номер договора эквайринга', 'n_agr', 'contract_number'],
    'd_valid_from': ['Дата регистрации договора', 'Дата начала договора', 'd_valid_from', 'Дата начала'],
    'd_valid_to': ['Дата закрытия договора', 'Дата окончания договора', 'd_valid_to', 'Дата окончания'],
    'tariff': ['Тариф', 'Тарифный план', 'Тариф клиента', 'Название тарифа', 'Наименование тарифа', 'tariff_name'],
    'trx_sum': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
    'commission_from_ops': [
        'Комиссия (% с операций)',
        'Комиссия \n(% с операций)',
        'Комиссия % с операций',
        'Количество (% с операций)',
        'Комиссия эквайринга',
    ],
    'commission_monthly': [
        'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)',
        'Комиссия CN (₽ в месяц)',
        'Комиссия в месяц',
        'Комиссия (руб в месяц)',
        'Комиссия (Р в месяц)',
    ],
    'filial': ['Филиал', 'Региональный филиал', 'Филиал договора', 'branch_nm', 'filial_rf'],
}

In [ ]:
frames_all = []

for src in excel_sources:
    path = Path(src['path'])
    if not path.exists():
        raise FileNotFoundError(f'Нет файла: {path}')

    raw = pd.read_excel(path, header=src['header'])
    resolved = {k: pick_column(raw.columns, aliases) for k, aliases in COL_ALIASES.items()}
    required = ['agr_id', 'tariff', 'trx_sum', 'commission_monthly', 'filial', 'd_valid_from']
    missing = [k for k in required if resolved[k] is None]
    if missing:
        raise ValueError(f"{path.name}: не найдены колонки {missing}. Доступные: {list(raw.columns)[:40]}")

    df = pd.DataFrame({
        'agr_id': raw[resolved['agr_id']].map(normalize_agr_id),
        'company_name': raw[resolved['company_name']] if resolved['company_name'] else None,
        'inn': raw[resolved['inn']].map(normalize_inn) if resolved['inn'] else None,
        'contract_number': raw[resolved['contract_number']] if resolved['contract_number'] else None,
        'd_valid_from': pd.to_datetime(raw[resolved['d_valid_from']], errors='coerce'),
        'd_valid_to': raw[resolved['d_valid_to']] if resolved['d_valid_to'] else np.nan,
        'tariff_raw': raw[resolved['tariff']],
        'trx_sum': to_num(raw[resolved['trx_sum']]),
        'commission_from_ops': to_num(raw[resolved['commission_from_ops']]) if resolved['commission_from_ops'] else np.nan,
        'commission_monthly': to_num(raw[resolved['commission_monthly']]),
        'filial_raw': raw[resolved['filial']],
    })
    df['report_month'] = pd.to_datetime(src['report_month'])
    df['report_month_str'] = df['report_month'].dt.strftime('%Y-%m')
    df['tariff_group'] = df['tariff_raw'].map(classify_tariff)
    df['filial_norm'] = (
        df['filial_raw'].astype(str)
        .str.replace('\xa0', ' ', regex=False)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )
    df['is_spb'] = df['filial_norm'].str.contains(TARGET_RF, case=False, na=False)
    df = df[df['agr_id'].notna()].copy()
    frames_all.append(df)
    print(f"{src['report_month'][:7]}: raw={len(raw):,} | with agr_id={len(df):,} | SPB={df['is_spb'].sum():,}")

excel_all = pd.concat(frames_all, ignore_index=True)
check_ids = set(CHECK_AGR_IDS)
excel_check = excel_all[excel_all['agr_id'].isin(check_ids)].copy()
print('Строк по check agr_id:', len(excel_check))
print('Найдено уникальных agr_id:', excel_check['agr_id'].nunique(), '/', len(CHECK_AGR_IDS))

In [ ]:
rows = []
for agr in CHECK_AGR_IDS:
    sub = excel_check[excel_check['agr_id'] == agr]
    months = sorted(sub['report_month_str'].dropna().unique().tolist())
    months_spb = sorted(sub.loc[sub['is_spb'], 'report_month_str'].dropna().unique().tolist())
    tariffs = sorted({str(x) for x in sub['tariff_group'].dropna().unique()})
    tariffs_raw = sorted({str(x).strip() for x in sub['tariff_raw'].dropna().unique() if str(x).strip()})
    filials = sorted({str(x) for x in sub['filial_norm'].dropna().unique()})

    in_all_months = set(months) == set(MONTHS)
    in_all_months_spb = set(months_spb) == set(MONTHS)
    stable_tariff = len(tariffs) == 1 and tariffs[0] in TARIFF_ORDER
    tariff_group = tariffs[0] if stable_tariff else (', '.join(tariffs) if tariffs else None)

    june = sub[sub['report_month_str'] == FILTER_MONTH]
    if len(june):
        d_valid_from = june['d_valid_from'].min()
        has_close = any(_is_closed(x) for x in june['d_valid_to'])
        trx_sum_june = float(np.nansum([_num0(x) for x in june['trx_sum']]))
        cm_june = float(np.nansum([_num0(x) for x in june['commission_monthly']]))
        company = june['company_name'].dropna().astype(str).iloc[0] if june['company_name'].notna().any() else None
        inn = june['inn'].dropna().astype(str).iloc[0] if june['inn'].notna().any() else None
        contract = june['contract_number'].dropna().astype(str).iloc[0] if june['contract_number'].notna().any() else None
    else:
        d_valid_from = pd.NaT
        has_close = None
        trx_sum_june = np.nan
        cm_june = np.nan
        company = None
        inn = None
        contract = None

    fail_reasons = []
    if sub.empty:
        fail_reasons.append('нет в Excel апр–июнь')
    else:
        if not in_all_months:
            fail_reasons.append(f'не во всех месяцах ({months})')
        if not any(sub['is_spb']):
            fail_reasons.append('не СПб РФ')
        elif not in_all_months_spb:
            fail_reasons.append(f'СПб не во всех месяцах ({months_spb})')
        if not stable_tariff:
            fail_reasons.append(f'нет одного целевого тарифа ({tariffs_raw or tariffs})')

    pass_june = None
    if stable_tariff and in_all_months_spb and tariff_group in ('Стандарт', 'Меню возможностей') and len(june):
        ok_reg = pd.notna(d_valid_from) and d_valid_from <= REG_DATE_MAX
        ok_open = not has_close
        if tariff_group == 'Стандарт':
            ok_trx = (trx_sum_june > 0) and (trx_sum_june <= STANDARD_TRX_SUM_MAX)
        else:
            ok_trx = trx_sum_june <= MENU_TRX_SUM_MAX
        ok_cm = abs(cm_june) <= COMM_MONTHLY_EPS
        pass_june = bool(ok_reg and ok_open and ok_trx and ok_cm)
        if not ok_reg:
            fail_reasons.append(f'июнь: d_valid_from={d_valid_from} > {REG_DATE_MAX.date()} или пусто')
        if not ok_open:
            fail_reasons.append('июнь: есть дата закрытия')
        if not ok_trx:
            fail_reasons.append(f'июнь: trx_sum={trx_sum_june}')
        if not ok_cm:
            fail_reasons.append(f'июнь: commission_monthly={cm_june}')
    elif stable_tariff and in_all_months_spb and tariff_group in ('0', 'По Акту индивидуальный'):
        pass_june = True  # доп. июньских фильтров нет

    eligible = (
        in_all_months_spb
        and stable_tariff
        and (pass_june is True if tariff_group in ('Стандарт', 'Меню возможностей', '0', 'По Акту индивидуальный') else False)
    )

    rows.append({
        'agr_id': agr,
        'found': not sub.empty,
        'months': ', '.join(months) if months else None,
        'months_spb': ', '.join(months_spb) if months_spb else None,
        'is_spb_any': bool(sub['is_spb'].any()) if len(sub) else False,
        'in_all_3_months': in_all_months,
        'in_all_3_months_spb': in_all_months_spb,
        'tariff_group': tariff_group,
        'tariff_raw': ', '.join(tariffs_raw) if tariffs_raw else None,
        'filial': '; '.join(filials[:3]) if filials else None,
        'company_name': company,
        'inn': inn,
        'contract_number': contract,
        'd_valid_from_june': d_valid_from,
        'has_close_date_june': has_close,
        'trx_sum_june': trx_sum_june,
        'commission_monthly_june': cm_june,
        'pass_june_filters': pass_june,
        'eligible_like_selection': eligible,
        'fail_reasons': '; '.join(fail_reasons) if fail_reasons else 'OK',
    })

diag_df = pd.DataFrame(rows)
display(diag_df)

print('\nИтог:')
print('найдено:', int(diag_df['found'].sum()), '/', len(diag_df))
print('eligible_like_selection:', int(diag_df['eligible_like_selection'].sum()), '/', len(diag_df))
display(diag_df[['agr_id', 'eligible_like_selection', 'fail_reasons']])

In [ ]:
# Детализация по месяцам (как в итоговых листах отбора)
detail_cols = [
    'agr_id', 'report_month_str', 'is_spb', 'company_name', 'inn', 'contract_number',
    'd_valid_from', 'd_valid_to', 'tariff_raw', 'tariff_group',
    'trx_sum', 'commission_from_ops', 'commission_monthly', 'filial_norm',
]
detail = (
    excel_check[detail_cols]
    .sort_values(['agr_id', 'report_month_str'])
    .reset_index(drop=True)
)
display(detail)

missing = [a for a in CHECK_AGR_IDS if a not in set(excel_check['agr_id'])]
if missing:
    print('Нет в Excel апр–июнь:', missing)